# Colab 02 — Entrenamiento final (Entregable 3)
**Requisitos previos:** `colab_01_hpo.ipynb` terminado (`resultados/hpo/mejor_config.json`) y `docs/protocolo.md` en Drive.

- **Etapa A:** 11 modelos × 5 semillas (55 corridas, ~9 h → 2 sesiones) con la mejor configuración de la búsqueda (`unet` usa la del protocolo).
- **Elección del top 3** por validación (`elegir_top3.py`).
- **Etapa B:** 5 semillas extra para el modelo original y el top 3 (20 corridas, ~3,5 h) → 10 semillas.
- Reanudable: si se corta, **repite la misma celda**. No lee el test. nnU-Net va en `colab_02b_nnunet.ipynb`.

In [ ]:
# Celda 1 — GPU
import os
!nvidia-smi | head -15

In [ ]:
# Celda 2 — dependencias
%pip install -q segmentation-models-pytorch monai einops timm scipy scikit-image pandas opencv-python-headless tqdm pytest matplotlib onnx onnxruntime
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
assert torch.cuda.is_available(), "Sin GPU: activa una GPU (Runtime → Change runtime type)"

In [ ]:
# Celda 3 — montar Drive y fijar rutas
from google.colab import drive
drive.mount('/content/drive')
CARPETA_DRIVE = "/content/drive/MyDrive/proyecto_tb"      # AJUSTAR si cambia
RAIZ_LOCAL = "/content/proyecto_tb"                        # copia local (rápida)
RES_HPO = f"{CARPETA_DRIVE}/resultados"                    # donde quedó la búsqueda (hpo/)
RES_E3 = f"{CARPETA_DRIVE}/resultados_e3"                  # resultados finales del Entregable 3
HPO_CFG = f"{RES_HPO}/hpo/mejor_config.json"
SPL = f"{RAIZ_LOCAL}/datos/splits"
os.makedirs(RES_E3, exist_ok=True)
assert os.path.isdir(f"{CARPETA_DRIVE}/codigo"), "Falta codigo/ en Drive"
assert os.path.isdir(f"{CARPETA_DRIVE}/datos/splits"), "Falta datos/splits"
print("Drive OK:", CARPETA_DRIVE, "\nResultados E3:", RES_E3)

In [ ]:
# Celda 4 — copiar código y datos a disco local: SOLO train/val (el test no se copia)
import csv, shutil
def a_local(r, raiz):
    n = r.replace("\\", "/"); i = n.find("datos/"); assert i >= 0, r
    return os.path.join(raiz, n[i:])
shutil.copytree(f"{CARPETA_DRIVE}/codigo", f"{RAIZ_LOCAL}/codigo", dirs_exist_ok=True)
os.makedirs(SPL, exist_ok=True)
for nombre in ("train", "val"):
    p = f"{CARPETA_DRIVE}/datos/splits/{nombre}.csv"
    if not os.path.exists(p): print("  (no existe", nombre, ")"); continue
    filas = list(csv.DictReader(open(p, newline="", encoding="utf-8")))
    campos = list(filas[0].keys())
    for f in filas:
        dst, src = a_local(f["ruta"], RAIZ_LOCAL), a_local(f["ruta"], CARPETA_DRIVE)
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy2(src, dst)
        f["ruta"] = dst
    with open(f"{SPL}/{nombre}.csv", "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=campos)
        w.writeheader(); w.writerows(filas)
    print(f"  {nombre:16s} {len(filas)} filas")
for mf in ("split_manifest_v2.json", "split_manifest.json"):
    if os.path.exists(f"{CARPETA_DRIVE}/datos/splits/{mf}"):
        shutil.copy2(f"{CARPETA_DRIVE}/datos/splits/{mf}", f"{SPL}/{mf}")

import sys; sys.path.insert(0, f"{RAIZ_LOCAL}/codigo")

In [ ]:
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --recomponer-resumen --resultados "{RES_E3}"

In [ ]:
# Celda — congelar / verificar protocolo.md (SHA-256)
import hashlib
p = f"{CARPETA_DRIVE}/docs/protocolo.md"
assert os.path.exists(p), "Falta docs/protocolo.md en Drive"
h = hashlib.sha256(open(p, "rb").read()).hexdigest()
f = f"{RES_E3}/protocolo.sha256"
if os.path.exists(f):
    assert open(f).read().split()[0] == h, "protocolo.md CAMBIÓ después de congelarse: registra una enmienda"
    print("protocolo verificado:", h[:16], "…")
else:
    open(f, "w").write(h + "  protocolo.md\n"); print("protocolo CONGELADO:", h)

In [ ]:
# Celda — pruebas unitarias del modelo original (control antes de gastar GPU)
!cd "{RAIZ_LOCAL}" && python -m pytest codigo/tests -q 2>&1 | tail -4

In [ ]:
# Celda 5 — plan de la Etapa A (no entrena)
assert os.path.exists(HPO_CFG), "Falta resultados/hpo/mejor_config.json: termina colab_01_hpo"
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --dry-run --modelos unet unet_fuerte attention_unet unet_pp deeplabv3p transunet swin_unet unetr segformer mednext lacca_unet --config-hpo "{HPO_CFG}" --splits-dir "{SPL}" --resultados "{RES_E3}" | tail -25

In [ ]:
# Celda 6 — ETAPA A: 11 modelos x 5 semillas (corta limpio a 330 min; repite hasta "Nada pendiente")
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --modelos unet unet_fuerte attention_unet unet_pp deeplabv3p transunet swin_unet unetr segformer mednext lacca_unet --config-hpo "{HPO_CFG}" --splits-dir "{SPL}" --resultados "{RES_E3}" --max-minutos 330

In [ ]:
# Celda 7 — elegir el top 3 por validación (falla si faltan semillas de la Etapa A)
!python "{RAIZ_LOCAL}/codigo/elegir_top3.py" --resultados "{RES_E3}"

In [ ]:
# Celda 8 — ETAPA B: semillas extra para el original y el top 3 (repite hasta "Nada pendiente")
import json
top = json.load(open(f"{RES_E3}/top3.json"))["top"]
modelos_b = " ".join(["lacca_unet"] + top)
print("Etapa B:", modelos_b)
!python "{RAIZ_LOCAL}/codigo/driver_entrenamiento.py" --modelos {modelos_b} --semillas 7 11 99 1234 9999 --config-hpo "{HPO_CFG}" --splits-dir "{SPL}" --resultados "{RES_E3}" --max-minutos 330

In [ ]:
# Celda 9 — auditoría de resultados (duplicados, modelos idénticos, conteos)
!python "{RAIZ_LOCAL}/codigo/limpiar_resultados.py" "{RES_E3}/resumen_runs.csv"

## Al terminar
Si la celda 9 muestra pares con Dice idéntico, **avísame antes de seguir** (sería otro duplicado). Siguiente: `colab_02b_nnunet.ipynb` y `colab_03_ablaciones.ipynb`.